In [1]:
import time
import os
import joblib
import pandas as pd
import numpy as np
import pyswarms as ps
from tensorflow.keras.models import load_model

In [6]:
# ----------------------------------------------------
# 0. 전역 설정 및 파일 로드 경로
# ----------------------------------------------------
N_ENSEMBLE = 10
TARGET_NAMES = ["Deflection(mm)", "Weight(kg)"]
MODEL_PATHS = [f'saved_models/pv_ensemble_model_{i+1}.keras' for i in range(N_ENSEMBLE)]
SCALER_DIR = '.' # 스케일러 저장 경로

# ----------------------------------------------------
# 1. 앙상블 모델 및 스케일러 로딩
# ----------------------------------------------------
try:
    ENSEMBLE_MODELS = [load_model(path, compile=False) for path in MODEL_PATHS]
    X_SCALER_LOADED = joblib.load(os.path.join(SCALER_DIR, 'scaler_X_pvmodule.joblib'))
    Y_SCALER_LOADED = joblib.load(os.path.join(SCALER_DIR, 'scaler_y_pvmodule.joblib'))
except Exception as e:
    print(f"오류: 모델 또는 스케일러 로드 실패. 저장 경로 확인 필요. ({e})")
    raise

print("✅ 앙상블 모델 및 스케일러 로딩 완료.")

✅ 앙상블 모델 및 스케일러 로딩 완료.


C:\Users\admin\anaconda3\envs\py31010_auto\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator RobustScaler from version 1.5.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [10]:
CONSTRAINTS = {
    "Deflection(mm)": {"min": None, "max": 12.3}, 
    "Weight(kg)": {"min": None, "max": 3.6}       
}

PENALTY_CONST = 100.0

In [11]:
# ----------------------------------------------------
# 2. 제약 조건 반영 목적 함수 정의 (PSO용)
# ----------------------------------------------------
def objective_for_PSO_constrained(X):
    """
    Pyswarms에 전달되는 목적 함수. 
    입력 X는 (n_particles, dimensions) 형태의 numpy 배열입니다.
    """
    n_particles = X.shape[0]
    F_obj_values = []
    
    # 변위(음수)는 0에 가깝도록 "최대화(+1)", 무게(양수)는 "최소화(-1)"
    GOAL_SIGNS = np.array([1, -1]) 
    WEIGHTS = np.array([1.0, 1.0])
    
    for i in range(n_particles):
        # 5개 인자 언패킹
        a, b, c, d, e = X[i, :]
        X_input = np.array([[a, b, c, d, e]])
        X_scaled = X_SCALER_LOADED.transform(X_input)
        
        # 앙상블 평균 예측 (원본 물리적 값 반환)
        preds_raw = [model.predict(X_scaled, verbose=0)[0] for model in ENSEMBLE_MODELS]
        y_pred_mean_raw = np.mean(np.array(preds_raw), axis=0)

        # 공평한 F_obj 계산을 위한 스케일링
        y_pred_mean_scaled = Y_SCALER_LOADED.transform(y_pred_mean_raw.reshape(1, -1))[0]
        
        # F_obj (벌칙 미적용)
        F_obj_unpenalized = np.sum(GOAL_SIGNS * WEIGHTS * y_pred_mean_scaled)
        
        # 제약 조건 검사 및 벌칙 적용 (원본 물리적 값 기준)
        penalty = 0.0
        for j, target_name in enumerate(TARGET_NAMES):
            value = y_pred_mean_raw[j]
            
            # 처짐량은 절댓값으로 변환하여 검사
            if target_name == "Deflection(mm)":
                check_value = abs(value)
            else:
                check_value = value
                
            if target_name in CONSTRAINTS:
                max_req = CONSTRAINTS[target_name]["max"]
                if max_req is not None and check_value > max_req:
                    penalty += PENALTY_CONST * (check_value - max_req)
            
        F_obj_constrained = F_obj_unpenalized - penalty
        F_obj_values.append(F_obj_constrained)
        
    # PSO는 내부적으로 '최소화' 알고리즘이므로, 목적 함수 최대화를 위해 -1을 곱함
    return -np.array(F_obj_values)

# ----------------------------------------------------
# 3. Particle Swarm Optimization (PSO) 실행
# ----------------------------------------------------
if __name__ == '__main__':
    start_total = time.time()
    
    # PV 프레임 입력 인자 범위 (5차원)
    INPUT_LB = np.array([1.5, 8.0, 1.5, 1.7, 2.5]) 
    INPUT_UB = np.array([2.5, 14.0, 2.5, 3.0, 5.0]) 
    bounds = (INPUT_LB, INPUT_UB)

    options = {
        'c1': 0.5, 
        'c2': 0.3, 
        'w': 0.9   
    }

    N_PARTICLES = 30
    N_ITERS = 15     # 30 * 15 = 총 450회 탐색 (BO, DE와 동일 조건)

    optimizer_pso = ps.single.GlobalBestPSO(
        n_particles=N_PARTICLES,
        dimensions=5, # 5차원
        options=options,
        bounds=bounds,
        init_pos=None 
    )

    print(f"\n--- 제약 조건 반영 최적 입력 인자 탐색 시작 (PSO, Total Calls: {N_PARTICLES * N_ITERS}) ---")

    cost, pos = optimizer_pso.optimize(objective_for_PSO_constrained, iters=N_ITERS)

    # ----------------------------------------------------
    # 4. 최적 입력 인자 및 예측 결과 출력 (PSO 결과)
    # ----------------------------------------------------
    best_X_params_pso = {
        'a': pos[0], 'b': pos[1], 'c': pos[2], 'd': pos[3], 'e': pos[4]
    }
    
    max_F_obj_pso = -cost # -1을 다시 곱해 원래 스케일 복원
    
    print("\n=== 제약 조건 만족 복합 목표 최대화 최적 입력 인자 조합 (PSO) ===")
    for k, v in best_X_params_pso.items():
        print(f"    {k}: {v:.4f}")
    
    print(f"최대 목적 함수 값 (F_obj): {max_F_obj_pso:.4f}")

    # 최적 X_best 예측
    X_optimal_pso = np.array([[pos[0], pos[1], pos[2], pos[3], pos[4]]])
    X_optimal_scaled_pso = X_SCALER_LOADED.transform(X_optimal_pso)
    
    # 모델이 원본 값을 주므로 inverse_transform 삭제
    preds_raw_optimal_pso = [model.predict(X_optimal_scaled_pso, verbose=0)[0] for model in ENSEMBLE_MODELS]
    y_pred_mean_optimal_pso = np.mean(np.array(preds_raw_optimal_pso), axis=0)

    print("\n--- 최적 해의 예측된 출력값 (Y) ---")
    for name, value in zip(TARGET_NAMES, y_pred_mean_optimal_pso):
        direction = "⬇️ (Minimize Goal)" if name == "Weight(kg)" else "⬆️ (Maximize closer to 0)"
        
        # 제약 조건 만족 여부 표시 (절댓값 처리 재적용)
        check_value = abs(value) if name == "Deflection(mm)" else value
        constraint_info = ""
        
        if name in CONSTRAINTS:
             max_req = CONSTRAINTS[name]['max']
             if max_req is not None and check_value > max_req:
                constraint_info = "❌ Violated"
             else:
                constraint_info = "✅ Satisfied"
                
        print(f"    - **{name}:** {value:.4f} {direction} ({constraint_info})")

    # ----------------------------------------------------
    # 5. 딥 앙상블의 불확실성 분석 (X_best 지점)
    # ----------------------------------------------------
    preds_raw_arr = np.array(preds_raw_optimal_pso) 
    
    # 모델 자체가 원래 스케일의 값을 내보내므로, scale_을 곱해줄 필요 없이 바로 사용 가능
    mean_original = np.mean(preds_raw_arr, axis=0)
    std_original = np.std(preds_raw_arr, axis=0)

    print("\n=== 📊 앙상블 예측 신뢰성 분석 (X_best 지점) ===")
    print("  (95% 예측 구간: Mean ± 1.96 * Std)")

    results_df = pd.DataFrame({
        'Output': TARGET_NAMES,
        'Mean (Pred)': mean_original,
        'Std (Uncertainty)': std_original,
        'Lower 95% PI': mean_original - 1.96 * std_original,
        'Upper 95% PI': mean_original + 1.96 * std_original
    })

    print(results_df.to_markdown(floatfmt=".4f"))

    # ----------------------------------------------------
    # 6. 반복 검증 (Local Verification)
    # ----------------------------------------------------
    print("\n--- 국소적 검증(Local Verification)을 위한 주변 샘플링 시작 ---")

    N_LOCAL_SAMPLES = 10 
    TOLERANCE_FACTOR = 0.01 
    local_f_obj_values = []

    for i in range(N_LOCAL_SAMPLES):
        X_local_sample = []
        for key in ['a', 'b', 'c', 'd', 'e']:
            mean_val = best_X_params_pso[key]
            low = mean_val * (1 - TOLERANCE_FACTOR)
            high = mean_val * (1 + TOLERANCE_FACTOR)
            X_local_sample.append(np.random.uniform(low, high))
            
        X_input_for_obj = np.array([X_local_sample]) 
        cost_val = objective_for_PSO_constrained(X_input_for_obj)[0]
        local_f_obj_values.append(-cost_val)

    print("\n=== 국소적 검증 (Local Verification) 결과 ===")
    print(f"기준점 (X_best) F_obj: **{max_F_obj_pso:.4f}**")
    print(f"주변 샘플 ({N_LOCAL_SAMPLES}개) F_obj 범위: **{np.min(local_f_obj_values):.4f} ~ {np.max(local_f_obj_values):.4f}**")
    print(f"주변 샘플 F_obj 평균: **{np.mean(local_f_obj_values):.4f}**")
    
    print(f"\n✅ PSO 최적화 및 평가 완료. (총 소요 시간: {time.time() - start_total:.2f}초)")

2026-05-27 10:39:31,440 - pyswarms.single.global_best - INFO - Optimize for 15 iters with {'c1': 0.5, 'c2': 0.3, 'w': 0.9}



--- 제약 조건 반영 최적 입력 인자 탐색 시작 (PSO, Total Calls: 450) ---


pyswarms.single.global_best: 100%|█████████████████████████████████████████████████████████████|15/15, best_cost=-0.236
2026-05-27 10:43:52,033 - pyswarms.single.global_best - INFO - Optimization finished | best cost: -0.23565423488616943, best pos: [ 1.58343772 10.59372375  1.50022095  2.72741066  4.0128183 ]



=== 제약 조건 만족 복합 목표 최대화 최적 입력 인자 조합 (PSO) ===
    a: 1.5834
    b: 10.5937
    c: 1.5002
    d: 2.7274
    e: 4.0128
최대 목적 함수 값 (F_obj): 0.2357

--- 최적 해의 예측된 출력값 (Y) ---
    - **Deflection(mm):** -11.4640 ⬆️ (Maximize closer to 0) (✅ Satisfied)
    - **Weight(kg):** 3.5916 ⬇️ (Minimize Goal) (✅ Satisfied)

=== 📊 앙상블 예측 신뢰성 분석 (X_best 지점) ===
  (95% 예측 구간: Mean ± 1.96 * Std)
|    | Output         |   Mean (Pred) |   Std (Uncertainty) |   Lower 95% PI |   Upper 95% PI |
|---:|:---------------|--------------:|--------------------:|---------------:|---------------:|
|  0 | Deflection(mm) |      -11.4640 |              0.0724 |       -11.6059 |       -11.3220 |
|  1 | Weight(kg)     |        3.5916 |              0.0382 |         3.5167 |         3.6664 |

--- 국소적 검증(Local Verification)을 위한 주변 샘플링 시작 ---

=== 국소적 검증 (Local Verification) 결과 ===
기준점 (X_best) F_obj: **0.2357**
주변 샘플 (10개) F_obj 범위: **-0.0089 ~ 0.2518**
주변 샘플 F_obj 평균: **0.2163**

✅ PSO 최적화 및 평가 완료. (총 소요 시간: 267.33초)
